### Silver – stores

#### Purpose
Transform the Bronze `stores` table into a clean and analytics-ready
Silver table by:
- Standardizing column names using a reusable UDF
- Enforcing correct data types
- Handling nulls based on business rules
- Deduplicating records
- Isolating malformed records into a quarantine table

#### Source
- coffee.bronze.stores

#### Targets
- coffee.silver.stores
- coffee.silver.quarantine_stores


In [0]:
%python
# Notebook Configuration using Databricks Widgets
# Purpose:
# - Avoid hard coding environment-specific values
# - Make the code reusable across tables and environments
dbutils.widgets.text("catalog", "coffee")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("source_table", "stores")
dbutils.widgets.text("default_watermark", "1900-01-01")

# Read widget values into Python variables
# These variables are used throughout the notebook
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
source_table = dbutils.widgets.get("source_table")
default_watermark = dbutils.widgets.get("default_watermark")


In [0]:
%run ./Silver_utils/silver_transform_utils

In [0]:
%python
df_bronze = spark.table(f"{catalog}.{bronze_schema}.{source_table}")


In [0]:
%python
# Standardize all incoming column names using the central UDF.
# This ensures consistent snake_case naming across all Silver tables,
# regardless of how the raw files were named in Bronze.

df_std = standardize_columns(df_bronze)



In [0]:
%python
df_std.createOrReplaceTempView(f"bronze_{source_table}_std")


In [0]:
%python
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{silver_schema}.{source_table} (
  store_id INT,
  store_name STRING,
  street STRING,
  city STRING,
  state STRING,
  postal_code STRING,
  latitude DOUBLE,
  longitude DOUBLE,

  -- Bronze metadata
  loaded_at TIMESTAMP,
  updated_at TIMESTAMP,
  load_dt DATE,
  source_file STRING,
  source_table STRING,

  -- Silver audit columns
  silver_loaded_at TIMESTAMP,
  silver_updated_at TIMESTAMP
)
USING DELTA
""")


In [0]:
CREATE OR REPLACE TEMP VIEW bronze_stores_incremental AS
SELECT *
FROM bronze_stores_std
WHERE loaded_at >
(
  SELECT COALESCE(MAX(loaded_at), '1900-01-01')
  FROM coffee.silver.stores
);


In [0]:
%python

# : Count invalid records for stores
# For stores, the minimum required columns are:
# store_id, store_name, city, state
# These are needed for a valid store dimension.

invalid_count = spark.sql("""
SELECT COUNT(*) AS cnt
FROM bronze_stores_incremental
WHERE
  store_id IS NULL
  OR store_name IS NULL
  OR city IS NULL
  OR state IS NULL
""").collect()[0]["cnt"]

print("Invalid store rows:", invalid_count)


In [0]:
%python

# Create and load stores quarantine table only if invalid rows exist


if invalid_count > 0:

    
    #Create stores quarantine table
   
    spark.sql("""
    CREATE TABLE IF NOT EXISTS  {catalog}.{silver_schema}.{source_table}_quarantine (
      store_id STRING,
      store_name STRING,
      street STRING,
      city STRING,
      state STRING,
      postal_code STRING,
      latitude STRING,
      longitude STRING,

      -- Bronze metadata
      loaded_at TIMESTAMP,
      updated_at TIMESTAMP,
      load_dt DATE,
      source STRING,
      source_file STRING,

      -- Quarantine metadata
      quarantine_reason STRING,
      quarantined_at TIMESTAMP
    )
    USING DELTA
    """)

   
    #  Merge invalid rows into quarantine (idempotent)

    spark.sql("""
    MERGE INTO  {catalog}.{silver_schema}.{source_table}_quarantine q
    USING (
      SELECT
        *,
        CASE
          WHEN store_id IS NULL THEN 'store_id is null'
          WHEN store_name IS NULL THEN 'store_name is null'
          WHEN city IS NULL THEN 'city is null'
          WHEN state IS NULL THEN 'state is null'
          ELSE 'unknown validation failure'
        END AS quarantine_reason,
        current_timestamp() AS quarantined_at
      FROM bronze_stores_incremental
      WHERE
        store_id IS NULL
        OR store_name IS NULL
        OR city IS NULL
        OR state IS NULL
    ) b
    ON q.store_id = b.store_id
    AND q.quarantine_reason = b.quarantine_reason
    WHEN NOT MATCHED THEN
    INSERT *;
    """)

else:
    print("No invalid store rows found. Quarantine table not created.")


In [0]:
%python
spark.sql(f""" 
MERGE INTO  {catalog}.{silver_schema}.{source_table} s
USING (

 
  -- Prepare clean, deduplicated incremental store records
  
  SELECT
    TRY_CAST(store_id AS INT)      AS store_id,
    store_name,
    street,
    city,
    state,
    postal_code,
    TRY_CAST(latitude AS DOUBLE)  AS latitude,
    TRY_CAST(longitude AS DOUBLE) AS longitude,

    loaded_at,
    updated_at,
    load_dt,
    source_file,
    'coffee.bronze.stores' AS source_table,

    current_timestamp() AS silver_updated_at

  FROM (
  
    
  -- Deduplication logic:
-- Bronze may contain duplicates for the same business key.
-- We keep only the latest version of each record using:
--   ROW_NUMBER() OVER (PARTITION BY <business_key> ORDER BY updated_at DESC)
--
-- This ensures Silver contains a single clean record per business key.

    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY store_id
             ORDER BY updated_at DESC
           ) AS rn
    FROM bronze_stores_incremental

    
    -- Enforce mandatory fields for a valid store
    
    WHERE
      store_id IS NOT NULL
      AND store_name IS NOT NULL
      AND city IS NOT NULL
      AND state IS NOT NULL
  )

  WHERE rn = 1

) b


-- Match condition: one row per store in Silver

ON s.store_id = b.store_id

-- CASE 1: Store already exists → UPDATE

WHEN MATCHED THEN
  UPDATE SET
    s.store_name        = b.store_name,
    s.street            = b.street,
    s.city              = b.city,
    s.state             = b.state,
    s.postal_code       = b.postal_code,
    s.latitude          = b.latitude,
    s.longitude         = b.longitude,
    s.updated_at        = b.updated_at,
    s.load_dt           = b.load_dt,
    s.source_file       = b.source_file,
    s.silver_updated_at = b.silver_updated_at


-- CASE 2: New store → INSERT

WHEN NOT MATCHED THEN
  INSERT (
    store_id,
    store_name,
    street,
    city,
    state,
    postal_code,
    latitude,
    longitude,
    loaded_at,
    updated_at,
    load_dt,
    source_file,
    source_table,
    silver_loaded_at,
    silver_updated_at
  )
  VALUES (
    b.store_id,
    b.store_name,
    b.street,
    b.city,
    b.state,
    b.postal_code,
    b.latitude,
    b.longitude,
    b.loaded_at,
    b.updated_at,
    b.load_dt,
    b.source_file,
    b.source_table,
    current_timestamp(),
    current_timestamp()
  )
  """)
